# Инструменты для работы с языком

... или зачем нужна предобработка. Источник: Высшая школа экономики.
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/10_Aehfbxgr3fxXPgI1gM5BTU8yOy-Z4U)

## Задача: классификация твитов по тональности

У нас есть датасет из твитов, про каждый указано, как он эмоционально окрашен: положительно или отрицательно. Задача: предсказывать эмоциональную окраску.

Классификацию по тональности используют в рекомендательных системах, чтобы понять, понравилось ли людям кафе, кино, etc.

Скачиваем куски датасета ([источник](http://study.mokoron.com/)): [положительные](https://www.dropbox.com/s/fnpq3z4bcnoktiv/positive.csv?dl=0), [отрицательные](https://www.dropbox.com/s/r6u59ljhhjdg6j0/negative.csv).

In [ ]:
# если у вас линукс / мак / collab или ещё какая-то среда, в которой работает wget, можно так:
!wget https://www.dropbox.com/s/fnpq3z4bcnoktiv/positive.csv
!wget https://www.dropbox.com/s/r6u59ljhhjdg6j0/negative.csv
!wget https://rusvectores.org/static/models/rusvectores4/unigrams/ruwikiruscorpora-nobigrams_upos_skipgram_300_5_2018.vec.gz

! git clone https://github.com/facebookresearch/fastText.git
! pip3 install fastText/. razdel natasha gensim

--2026-03-17 13:34:00--  https://www.dropbox.com/s/fnpq3z4bcnoktiv/positive.csv
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/6mg7rw3wltux83q2o4ah4/positive.csv?rlkey=cvruhzofza9kkfxwzyp2vskfd [following]
--2026-03-17 13:34:00--  https://www.dropbox.com/scl/fi/6mg7rw3wltux83q2o4ah4/positive.csv?rlkey=cvruhzofza9kkfxwzyp2vskfd
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc1a7b479a442c9b11a80fba4f40.dl.dropboxusercontent.com/cd/0/inline/C83TS3_a2P0_5ZY1Pp2mbEx3j2VMkGv6QzK1ifLFKcJdVjFe0Hw9Kmvx5H_ldVCZKBC1QaxhZZOcqyAE6YnCQlyujk1GWgmrFbsiHRF_tByVLhun3ZkIOFM4lS5W_7tmWZE/file# [following]
--2026-03-17 13:34:01--  https://uc1a7b479a442c9b11a80fba4f40.dl.dropboxusercontent.com/cd/0/inline/C83TS3_a2P0_5ZY1Pp2mbE

In [ ]:
import time
import string
from string import punctuation
from tqdm.notebook import tqdm

import pandas as pd
import numpy as np
import re

from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression # можно заменить на любимый классификатор
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from nltk import ngrams
import nltk
from nltk.tokenize import word_tokenize
from nltk import tokenize
from nltk import collocations
from nltk.corpus import stopwords

from bs4 import BeautifulSoup
import razdel
from natasha import Doc, MorphVocab, Segmenter, NewsEmbedding, NewsMorphTagger

import fasttext

import gensim

nltk.download('stopwords')
nltk.download('genesis')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package genesis to /root/nltk_data...
[nltk_data]   Unzipping corpora/genesis.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# считываем данные и заполняем общий датасет
positive = pd.read_csv('positive.csv', sep=';', usecols=[3], names=['text'])
positive['label'] = ['positive'] * len(positive)
negative = pd.read_csv('negative.csv', sep=';', usecols=[3], names=['text'])
negative['label'] = ['negative'] * len(negative)
df = pd.concat([positive, negative])

In [ ]:
df.head()

,text,label
0,"@first_timee хоть я и школота, но поверь, у на...",positive
1,"Да, все-таки он немного похож на него. Но мой ...",positive
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,positive
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",positive
4,@irina_dyshkant Вот что значит страшилка :D\nН...,positive


In [ ]:
df.label.value_counts()

,count
label,
positive,114911
negative,111923


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(df.text, df.label)

## Baseline: классификация необработанных n-грамм

### Векторизаторы

Что такое n-граммы?

N-грамма — последовательность из `n` элементов. С семантической точки зрения это может быть последовательность звуков, слогов, слов или букв. На практике чаще встречается N-грамма как ряд слов, устойчивые словосочетания называют коллокацией. Последовательность из двух последовательных элементов часто называют биграмма, последовательность из трёх элементов называется - триграмма. Не менее четырёх и выше элементов обозначается как N-грамма, N заменяется на количество последовательных элементов.

In [ ]:
sent = 'я знаю точно невозможное возможно'.split()
list(ngrams(sent, 1)) # униграммы

[('я',), ('знаю',), ('точно',), ('невозможное',), ('возможно',)]

In [ ]:
sent

['я', 'знаю', 'точно', 'невозможное', 'возможно']

In [ ]:
list(ngrams(sent, 2)) # биграммы

[('я', 'знаю'),
 ('знаю', 'точно'),
 ('точно', 'невозможное'),
 ('невозможное', 'возможно')]

In [ ]:
list(ngrams(sent, 3)) # триграммы

[('я', 'знаю', 'точно'),
 ('знаю', 'точно', 'невозможное'),
 ('точно', 'невозможное', 'возможно')]

In [ ]:
list(ngrams(sent, 5)) # ... пентаграммы?

[('я', 'знаю', 'точно', 'невозможное', 'возможно')]

Самый простой способ извлечь признаки из текстовых данных -- векторизаторы: `CountVectorizer` и `TfidfVectorizer`

Объект `CountVectorizer` делает простую вещь:
* строит для каждого документа (каждой пришедшей ему строки) вектор размерности `n`, где `n` -- количество слов или n-грам во всём корпусе
* заполняет каждый i-тый элемент количеством вхождений слова в данный документ

In [ ]:
vec = CountVectorizer(ngram_range=(1, 1))
bow = vec.fit_transform(x_train) # bow -- bag of words (мешок слов)

ngram_range отвечает за то, какие n-граммы мы используем в качестве признаков:<br/>
ngram_range=(1, 1) -- униграммы<br/>
ngram_range=(3, 3) -- триграммы<br/>
ngram_range=(1, 3) -- униграммы, биграммы и триграммы.

В vec.vocabulary_ лежит отображение слов к их индексам:

In [ ]:
list(vec.vocabulary_.items())[:10]

[('nikablacky', 63271),
 ('ты', 224405),
 ('другого', 128406),
 ('не', 165185),
 ('скажешь', 209426),
 ('своем', 207141),
 ('городе', 120989),
 ('rt', 74484),
 ('alisa550055', 11256),
 ('вот', 114697)]

In [ ]:
list(vec.vocabulary_.items())[-10:]

[('dmegvfqxxe', 26094),
 ('neropora', 62736),
 ('пижженом', 181786),
 ('4e5sb2okwo', 3489),
 ('yq6pldgt91', 94999),
 ('луиобажатель', 153878),
 ('2f61pnglfw', 2241),
 ('andronikovk8252', 12567),
 ('julia_bedlam', 43160),
 ('подавись', 184259)]

In [ ]:
len(vec.vocabulary_.items())

243423

In [ ]:
type(bow)

scipy.sparse._csr.csr_matrix

In [ ]:
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(bow, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
len(clf.coef_[0])

243423

In [ ]:
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.77      0.77      0.77     28219
    positive       0.77      0.77      0.77     28490

    accuracy                           0.77     56709
   macro avg       0.77      0.77      0.77     56709
weighted avg       0.77      0.77      0.77     56709



Попробуем сделать то же самое для триграмм:

In [ ]:
vec = CountVectorizer(ngram_range=(3, 3))
bow = vec.fit_transform(x_train)
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(bow, y_train)
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.47      0.72      0.57     18218
    positive       0.82      0.61      0.70     38491

    accuracy                           0.65     56709
   macro avg       0.64      0.66      0.63     56709
weighted avg       0.71      0.65      0.66     56709



In [ ]:
len(vec.vocabulary_.items())

1328683

(как вы думаете, почему в результатах теперь такой разброс по сравнению с униграммами?)

## TF-IDF векторизация

`TfidfVectorizer` делает то же, что и `CountVectorizer`, но в качестве значений – tf-idf каждого слова.

Как считается tf-idf:

TF (term frequency) – относительная частотность слова в документе:
$$ TF(t,d) = \frac{n_t}{\sum_k n_k} $$

`t` -- слово (term), `d` -- документ, $n_t$ -- количество вхождений слова, $n_k$ -- количество вхождений остальных слов

IDF (inverse document frequency) – обратная частота документов, в которых есть это слово:
$$ IDF(t, D) = \mbox{log} \frac{|D|}{|{d : t \in d}|} $$

`t` -- слово (term), `D` -- коллекция документов

Перемножаем их:
$$TFIDF_(t,d,D) = TF(t,d) \times IDF(i, D)$$

Сакральный смысл – если слово часто встречается в одном документе, но в целом по корпусу встречается в небольшом
количестве документов, у него высокий TF-IDF.

In [ ]:
vec = TfidfVectorizer(ngram_range=(1, 1))
bow = vec.fit_transform(x_train)
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(bow, y_train)
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.73      0.77      0.75     26722
    positive       0.79      0.75      0.77     29987

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



## Токенизация

Токенизировать -- значит, поделить текст на слова, или *токены*.

Самый наивный способ токенизировать текст -- разделить с помощью `split`. Но `split` упускает очень много всего, например, банально не отделяет пунктуацию от слов. Кроме этого, есть ещё много менее тривиальных проблем. Поэтому лучше использовать готовые токенизаторы.

In [ ]:
example = 'Но не каждый хочет что-то исправлять:('
word_tokenize(example)

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять', ':', '(']

В nltk вообще есть довольно много токенизаторов:

In [ ]:
dir(tokenize)[:16]

['BlanklineTokenizer',
 'LegalitySyllableTokenizer',
 'LineTokenizer',
 'MWETokenizer',
 'NLTKWordTokenizer',
 'PunktSentenceTokenizer',
 'PunktTokenizer',
 'RegexpTokenizer',
 'ReppTokenizer',
 'SExprTokenizer',
 'SpaceTokenizer',
 'StanfordSegmenter',
 'SyllableTokenizer',
 'TabTokenizer',
 'TextTilingTokenizer',
 'ToktokTokenizer']

Они умеют выдавать индексы начала и конца каждого токена:

In [ ]:
wh_tok = tokenize.WhitespaceTokenizer()
list(wh_tok.span_tokenize(example))

[(0, 2), (3, 5), (6, 12), (13, 18), (19, 25), (26, 38)]

(если вам было интересно, зачем вообще включать в модуль токенизатор, который работает как `.split()` :))

Некторые токенизаторы ведут себя специфично:

In [ ]:
tokenize.TreebankWordTokenizer().tokenize("don't stop me")

['do', "n't", 'stop', 'me']

Для некоторых задач это может быть полезно.

А некоторые -- вообще не для текста на естественном языке (не очень понятно, зачем это в nltk :)):

In [ ]:
tokenize.SExprTokenizer().tokenize("(a (b c)) d e (f)")

['(a (b c))', 'd', 'e', '(f)']

Самым быстрым токенизатором является TokTokTokenizer, оцените, на сколько он быстрее обработает обучающие данные по сравнению со стандартным TreebankWordTokenizer:

In [ ]:
start = time.time();
[tokenize.ToktokTokenizer().tokenize(tweet) for tweet in x_train]
end = time.time() - start
print(end)

7.786691427230835


In [ ]:
start = time.time();
tokenized_x_train = [tokenize.TreebankWordTokenizer().tokenize(tweet) for tweet in x_train]
end = time.time() - start
print(end)

10.072426319122314


ToktokTokenizer почти в 2 раза быстрее!

## PMI

Можно оценить взаимосвязь слов в корпусе и понять, какие биграммы наиболее часто встречаются в тексте. Для этого можно использовать метрику PMI (Pointwise Mutual Information) - поточечная взаимная информация. Метрика PMI для двух слов вычисляется по формуле:

$$pmi(x; y) = \log \frac{p(x,y)}{p(x)p(y)} = \log \frac{{p(x|y)}}{p(x)} = \log \frac{p(y|x)}{p(y)}$$

Здесь $p(y|x)$ - вероятность встретить слово $y$ после $x$, $p(y)$ - вероятность встретить слово $y$.

Оценим важность биграмм в нашем обучающем корпусе.

In [ ]:
word_tokenizer = tokenize.NLTKWordTokenizer()

raw_text = []

for comment in tqdm(x_train):
    raw_text.extend(word_tokenizer.tokenize(comment.lower()))

  0%|          | 0/170125 [00:00<?, ?it/s]

In [ ]:
bigram_finder = collocations.BigramCollocationFinder.from_words(raw_text)
bigram_measures = collocations.BigramAssocMeasures()

bigram_finder.apply_freq_filter(50)

pmi_score = bigram_finder.score_ngrams(bigram_measures.pmi)
pmi_score[:10]

[(('крайней', 'мере'), 15.047429567209294),
 (('черная', 'рамка'), 14.960570283708254),
 (('деда', 'мороза'), 14.662215827280496),
 (('//t.co/gmelsgmiuw', 'толи'), 14.629691655273259),
 (('серый', 'волк'), 14.604911331508344),
 (('иван', 'царевич'), 14.577181155066748),
 (('one', 'direction'), 14.565403987290395),
 (('официальный', 'трейлер'), 14.538543767215064),
 (('seconds', 'to'), 14.269513753304555),
 (('to', 'mars'), 14.254406860914347)]

In [ ]:
pmi_score[-10:]

[((':', ':'), -4.239992392134354),
 ((',', '@'), -4.362019060769143),
 (('я', ')'), -4.364890722929175),
 (('(', '!'), -4.982375286204437),
 (('я', '('), -4.988226609068459),
 ((',', ','), -5.272275709496064),
 ((',', '('), -5.635675434082835),
 ((':', ','), -5.657267889165119),
 ((')', ','), -5.98298895300119),
 (('(', ','), -6.447815418705815)]

## Стоп-слова и пунктуация

*Стоп-слова* -- это слова, которые часто встречаются практически в любом тексте и ничего интересного не говорят о конретном документе, то есть играют роль шума. Поэтому их принято убирать. По той же причине убирают и пунктуацию.

In [ ]:
print(stopwords.words('russian'))

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она', 'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее', 'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда', 'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до', 'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей', 'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем', 'была', 'сам', 'чтоб', 'без', 'будто', 'чего', 'раз', 'тоже', 'себе', 'под', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой', 'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее', 'сейчас', 'были', 'куда', 'зачем', 'всех', 'никогда', 'можно', 'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот', 'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту', 'моя', 'впр

In [ ]:
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [ ]:
noise = stopwords.words('russian') + list(punctuation)

В векторизаторах за стоп-слова, логичным образом, отвечает аргумент `stop_words`.

In [ ]:
vec = CountVectorizer(ngram_range=(1, 1), tokenizer=word_tokenize, stop_words=noise)
bow = vec.fit_transform(x_train)
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(bow, y_train)
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['``'] not in stop_words.
  warnings.warn(


              precision    recall  f1-score   support

    negative       0.80      0.77      0.79     29176
    positive       0.77      0.80      0.78     27533

    accuracy                           0.78     56709
   macro avg       0.78      0.78      0.78     56709
weighted avg       0.78      0.78      0.78     56709



In [ ]:
len(vec.vocabulary_.items())

259416

## Что ещё можно сделать?



Лемматизация – это сведение разных форм одного слова к начальной форме – *лемме*. Почему это хорошо?
* Во-первых, мы хотим рассматривать как отдельную фичу каждое *слово*, а не каждую его отдельную форму.
* Во-вторых, некоторые стоп-слова стоят только в начальной форме, и без лематизации выкидываем мы только её.

### [Mystem](https://tech.yandex.ru/mystem/)
Как с ним работать:
* можно скачать mystem и запускать [из терминала с разными параметрами](https://tech.yandex.ru/mystem/doc/)
* [pymystem3](https://pythonhosted.org/pymystem3/pymystem3.html) - обертка для питона, работает медленнее, но это удобно

Методы Mystem принимают строку, токенизатор вшит внутри. Можно, конечно, и пословно анализировать, но тогда он не сможет учитывать контекст.
Mystem умеет снимать омонимию по контексту

### [Pymorphy](http://pymorphy2.readthedocs.io/en/latest/)
Это модуль на питоне, довольно быстрый и с кучей функций.
pymorphy2 работает с отдельными словами. Если дать ему на вход предложение - он его просто не лемматизирует, т.к. не понимает
pymorphy2 берет на вход одно слово и соответственно вообще не умеет дизамбигуировать по контексту

### [Natasha](https://github.com/natasha/)

В библиотеке natasha реализовано множество полезных библиотек для русского языка: разбиение на токены и предложения, русскоязычные word embeddings, морфологический, синтаксический анализ, лемматизация, извлечение именованных сущностей и т.д. Модуль библиотеки Razdel, основанный на правилах, предназначен для разбиения текста на токены и предложения.
С помощью библиотеки natasaha можно также лемматизировать тексты.

In [ ]:
tokens = list(razdel.tokenize('Кружка-термос на 0.5л (50/64 см³, 516;...)'))
tokens

[Substring(0, 13, 'Кружка-термос'),
 Substring(14, 16, 'на'),
 Substring(17, 20, '0.5'),
 Substring(20, 21, 'л'),
 Substring(22, 23, '('),
 Substring(23, 28, '50/64'),
 Substring(29, 32, 'см³'),
 Substring(32, 33, ','),
 Substring(34, 37, '516'),
 Substring(37, 38, ';'),
 Substring(38, 41, '...'),
 Substring(41, 42, ')')]

In [ ]:
[_.text for _ in tokens]

['Кружка-термос',
 'на',
 '0.5',
 'л',
 '(',
 '50/64',
 'см³',
 ',',
 '516',
 ';',
 '...',
 ')']

In [ ]:
segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)

def natasha_lemmatize(text):
  doc = Doc(text)
  doc.segment(segmenter)
  doc.tag_morph(morph_tagger)
  for token in doc.tokens:
    token.lemmatize(morph_vocab)
  return {_.text: _.lemma for _ in doc.tokens}

text = 'Банковская гарантия подтверждает обязательства банка по погашению долга клиента перед третьей стороной. Документ, по сути, является обеспечением контракта и позволяет совершить сделку без привлечения оборотных или заемных средств. Предоставление гарантии — это платная услуга. Клиент выплачивает банку определенную комиссию.'

natasha_lemmatize(text)

{'Банковская': 'банковский',
 'гарантия': 'гарантия',
 'подтверждает': 'подтверждать',
 'обязательства': 'обязательство',
 'банка': 'банк',
 'по': 'по',
 'погашению': 'погашение',
 'долга': 'долг',
 'клиента': 'клиент',
 'перед': 'перед',
 'третьей': 'третий',
 'стороной': 'сторона',
 '.': '.',
 'Документ': 'документ',
 ',': ',',
 'сути': 'суть',
 'является': 'являться',
 'обеспечением': 'обеспечение',
 'контракта': 'контракт',
 'и': 'и',
 'позволяет': 'позволять',
 'совершить': 'совершить',
 'сделку': 'сделка',
 'без': 'без',
 'привлечения': 'привлечение',
 'оборотных': 'оборотный',
 'или': 'или',
 'заемных': 'заемный',
 'средств': 'средство',
 'Предоставление': 'предоставление',
 'гарантии': 'гарантия',
 '—': '—',
 'это': 'это',
 'платная': 'платный',
 'услуга': 'услуга',
 'Клиент': 'клиент',
 'выплачивает': 'выплачивать',
 'банку': 'банк',
 'определенную': 'определенный',
 'комиссию': 'комиссия'}

### Библиотека для анализа тональности [Dostoevsky](https://github.com/bureaucratic-labs/dostoevsky)

Библиотека для анализа тональности текстов на русском языке. Модель обучалась на наборе данных [RuSentiment](https://github.com/text-machine-lab/rusentiment).

# Embeddings  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PragmaticsLab/NLP-course-FinTech/blob/master/seminars/2/2_embeddings.ipynb)

## Word2Vec

Векторные модели, которые мы рассматривали до этого (tf-idf, BOW), условно называются *счётными*. Они основываются на том, что так или иначе "считают" слова и их соседей, и на основе этого строят вектора для слов.

Другой класс моделей, который более повсевмёстно распространён на сегодняшний день, называется *предсказательными* (или *нейронными*) моделями. Идея этих моделей заключается в использовании нейросетевых архитектур, которые "предсказывают" (а не считают) соседей слов. Одной из самых известных таких моделей является word2vec. Технология основана на нейронной сети, предсказывающей вероятность встретить слово в заданном контексте. Этот инструмент был разработан группой исследователей Google в 2013 году, руководителем проекта был Томаш Миколов (сейчас работает в Facebook). Вот две самые главные статьи:

* [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/pdf/1301.3781.pdf)
* [Distributed Representations of Words and Phrases and their Compositionality](https://arxiv.org/abs/1310.4546)


Полученные таким образом вектора называются *распределенными представлениями слов*, или **эмбеддингами**.


### Как это обучается?
Мы задаём вектор для каждого слова с помощью матрицы $w$ и вектор контекста с помощью матрицы $W$. По сути, word2vec является обобщающим названием для двух архитектур Skip-Gram и Continuous Bag-Of-Words (CBOW).  

**CBOW** предсказывает текущее слово, исходя из окружающего его контекста.

**Skip-gram**, наоборот, использует текущее слово, чтобы предугадывать окружающие его слова.

### Как это работает?
Word2vec принимает большой текстовый корпус в качестве входных данных и сопоставляет каждому слову вектор, выдавая координаты слов на выходе. Сначала он создает словарь, «обучаясь» на входных текстовых данных, а затем вычисляет векторное представление слов. Векторное представление основывается на контекстной близости: слова, встречающиеся в тексте рядом с одинаковыми словами (а следовательно, согласно дистрибутивной гипотезе, имеющие схожий смысл), в векторном представлении будут иметь близкие координаты векторов-слов. Для вычисления близости слов используется косинусное расстояние между их векторами.


С помощью дистрибутивных векторных моделей можно строить семантические пропорции (они же аналогии) и решать примеры:

* *король: мужчина = королева: женщина*
 $\Rightarrow$
* *король - мужчина + женщина = королева*

### Проблемы
Невозможно установить тип семантических отношений между словами: синонимы, антонимы и т.д. будут одинаково близки, потому что обычно употребляются в схожих контекстах. Поэтому близкие в векторном пространстве слова называют *семантическими ассоциатами*. Это значит, что они семантически связаны, но как именно — непонятно.


### RusVectōrēs


На сайте [RusVectōrēs](https://rusvectores.org/ru/) собраны предобученные на различных данных модели для русского языка, а также можно поискать наиболее близкие слова к заданному, посчитать семантическую близость нескольких слов и порешать примеры с помощью «калькулятором семантической близости».


Для других языков также можно найти предобученные модели — например, модели [fastText](https://fasttext.cc/docs/en/english-vectors.html) и [GloVe](https://nlp.stanford.edu/projects/glove/) (о них чуть дальше).

### Визуализация
А [вот тут](https://projector.tensorflow.org/) есть хорошая визуализация для английского.

## Gensim

Использовать предобученную модель эмбеддингов или обучить свою можно с помощью библиотеки `gensim`. Вот [ее документация](https://radimrehurek.com/gensim/models/word2vec.html).

### Как использовать готовую модель

Модели word2vec бывают разных форматов:

* .vec.gz — обычный файл
* .bin.gz — бинарник

Загружаются они с помощью одного и того же класса `KeyedVectors`, меняется только параметр `binary` у функции `load_word2vec_format`.

Если же эмбеддинги обучены **не** с помощью word2vec, то для загрузки нужно использовать функцию `load`. Т.е. для загрузки предобученных эмбеддингов *glove, fasttext, bpe* и любых других нужна именно она.

Скачаем с RusVectōrēs модель для русского языка, обученную на НКРЯ образца 2015 г.

In [ ]:
model_path = 'ruwikiruscorpora-nobigrams_upos_skipgram_300_5_2018.vec.gz'

model_ru = gensim.models.KeyedVectors.load_word2vec_format(model_path, binary=False)

Частеречные тэги нужны, поскольку это специфика скачанной модели - она была натренирована на словах, аннотированных их частями речи (и лемматизированных). **NB!** В названиях моделей на `rusvectores` указано, какой тегсет они используют (mystem, upos и т.д.)

Попросим у модели 10 ближайших соседей для каждого слова и коэффициент косинусной близости для каждого:

In [ ]:
words = ['день_NOUN', 'ночь_NOUN', 'человек_NOUN', 'семантика_NOUN', 'биткоин_NOUN']
for word in words:
    # есть ли слово в модели?
    if word in model_ru:
        print(word)
        # смотрим на вектор слова (его размерность 300, смотрим на первые 10 чисел)
        print(model_ru[word][:10])
        # выдаем 10 ближайших соседей слова:
        for word, sim in model_ru.most_similar(positive=[word], topn=10):
            # слово + коэффициент косинусной близости
            print(word, ': ', sim)
        print('\n')
    else:
        # Увы!
        print('Увы, слова "%s" нет в модели!' % word)

день_NOUN
[ 0.117177  0.008562 -0.054731  0.03821   0.006885  0.041716  0.063708
  0.070478  0.032087  0.050791]
неделя_NOUN :  0.7242119312286377
месяц_NOUN :  0.7178640365600586
утро_NOUN :  0.6738513708114624
вечер_NOUN :  0.6443344950675964
воскресенье_NOUN :  0.6362559795379639
час_NOUN :  0.632983386516571
накануне_ADV :  0.6304810047149658
днями_NOUN :  0.6276212930679321
днемя_NOUN :  0.621060848236084
ночь_NOUN :  0.6077756285667419


ночь_NOUN
[ 0.070529 -0.068594  0.029781  0.035559 -0.01488   0.072418 -0.01183
  0.051797 -0.024269  0.034406]
ночь_PROPN :  0.7884491682052612
вечер_NOUN :  0.7778281569480896
утро_NOUN :  0.7638111710548401
полночь_NOUN :  0.7437741756439209
рассвет_NOUN :  0.6889956593513489
полдень_NOUN :  0.6811894178390503
утро_PROPN :  0.6788180470466614
сумерки_NOUN :  0.6461666822433472
напроать_NOUN :  0.6451945900917053
напролет_VERB :  0.6393611431121826


человек_NOUN
[ 0.022094 -0.077399  0.038363 -0.051602  0.000347  0.073115 -0.068763
 -0.037081 

In [ ]:
# Находим косинусную близость пары слов:
print(model_ru.similarity('человек_NOUN', 'обезьяна_NOUN'))

0.32987356


Что получится, если вычесть из пиццы Италию и прибавить Сибирь?

* positive — вектора, которые мы складываем
* negative — вектора, которые вычитаем

In [ ]:
print(model_ru.most_similar(positive=['компьютер_NOUN', 'маленький_ADV'], negative=[], topn=10))

[('pc-совместимый_ADJ', 0.6711683869361877), ('однопроцессорный_ADJ', 0.6536186933517456), ('десктоп_NOUN', 0.6391705274581909), ('эмуляторовый_NOUN', 0.6382603645324707), ('zx80_NUM', 0.6370977759361267), ('нетбук_NOUN', 0.6319176554679871), ('ультрапортативный_ADJ', 0.6270133256912231), ('ноутбука_NOUN', 0.6220362186431885), ('powerbook_PROPN', 0.6181027293205261), ('mp3-плеер_NOUN', 0.6168028116226196)]


In [ ]:
model_ru.doesnt_match('пицца_NOUN пельмень_NOUN хот-дог_NOUN ананас_NOUN'.split())

'ананас_NOUN'

### Как обучить свою модель

Обучаем и сохраняем модель.


Основные параметры:
* данные должны быть итерируемым объектом
* size — размер вектора,
* window — размер окна наблюдения,
* min_count — мин. частотность слова в корпусе,
* sg — используемый алгоритм обучения (0 — CBOW, 1 — Skip-gram),
* sample — порог для downsampling'a высокочастотных слов,
* workers — количество потоков,
* alpha — learning rate,
* iter — количество итераций,
* max_vocab_size — позволяет выставить ограничение по памяти при создании словаря (т.е. если ограничение превышается, то низкочастотные слова будут выбрасываться). Для сравнения: 10 млн слов = 1Гб RAM.

**NB!** Обратите внимание, что тренировка модели не включает препроцессинг! Это значит, что избавляться от пунктуации, приводить слова к нижнему регистру, лемматизировать их, проставлять частеречные теги придется до тренировки модели (если, конечно, это необходимо для вашей задачи). Т.е. в каком виде слова будут в исходном тексте, в таком они будут и в модели.


In [ ]:
def text_to_wordlist(text, remove_stopwords=False):
    # оставляем только буквенные символы
    text = re.sub("[^а-яА-Яa-zA-Z0-9]", " ", text)
    # приводим к нижнему регистру и разбиваем на слова по символу пробела
    words = text.lower().split()
    if remove_stopwords: # убираем стоп-слова
        stops = stopwords.words("russian")
        words = [w for w in words if not w in stops]
    return (words)

sentences = []
print("Parsing sentences from training set...")
for text in x_train.values:
    sentences.append(text_to_wordlist(text))

Parsing sentences from training set...


In [ ]:
print("Training model...")

%time model_ru = gensim.models.word2vec.Word2Vec(sentences, workers=4, vector_size=300, min_count=10, window=10, sample=1e-3)

Training model...
CPU times: user 33.4 s, sys: 153 ms, total: 33.5 s
Wall time: 19.3 s


In [ ]:
# Смотрим, сколько в модели слов.
print(len(model_ru.wv.key_to_index))

16255


In [ ]:
def text_to_vector(text, model):
    words = text_to_wordlist(text)
    word_vectors = []
    for word in words:
        if word in model.wv:  # если слово есть в модели
            word_vectors.append(model.wv[word])
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(word_vectors, axis=0)  # усредняем векторы слов

# Преобразуем тексты в векторы
vec_train = np.array([text_to_vector(text, model_ru) for text in x_train.values])
vec_test = np.array([text_to_vector(text, model_ru) for text in x_test.values])

In [ ]:
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(vec_train, y_train)
pred = clf.predict(vec_test)
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.71      0.71      0.71     28027
    positive       0.71      0.71      0.71     28682

    accuracy                           0.71     56709
   macro avg       0.71      0.71      0.71     56709
weighted avg       0.71      0.71      0.71     56709



## FastText

FastText использует не только эмбеддинги слов, но и эмбеддинги n-грам. В корпусе каждое слово автоматически представляется в виде набора символьных n-грамм. Скажем, если мы установим n=3, то вектор для слова "where" будет представлен суммой векторов следующих триграм: "<wh", "whe", "her", "ere", "re>" (где "<" и ">" символы, обозначающие начало и конец слова). Благодаря этому мы можем также получать вектора для слов, отсутствуюших в словаре, а также эффективно работать с текстами, содержащими ошибки и опечатки.

* [Статья](https://aclweb.org/anthology/Q17-1010)
* [Сайт](https://fasttext.cc/)
* [Тьюториал](https://fasttext.cc/docs/en/support.html)
* [Вектора для 157 языков](https://fasttext.cc/docs/en/crawl-vectors.html)
* [Вектора, обученные на википедии](https://fasttext.cc/docs/en/pretrained-vectors.html) (отдельно для 294 разных языков)
* [Репозиторий](https://github.com/facebookresearch/fasttext)

Есть библиотека `fasttext` для питона (с готовыми моделями можно работать и через `gensim`).

In [ ]:
def prepare_fasttext_data(texts, labels, output_file):
    """Подготовка данных в формате FastText"""
    with open(output_file, 'w', encoding='utf-8') as f:
        for text, label in zip(texts, labels):
            # Очистка текста (можно добавить больше предобработки)
            text = text.replace('\n', ' ').strip()
            # Запись в формате FastText
            f.write(f'__label__{label} {text}\n')

prepare_fasttext_data(x_train.values, y_train.values, 'train.txt')
prepare_fasttext_data(x_test.values, y_test.values, 'test.txt')

model = fasttext.train_supervised(input='train.txt')
result = model.test('test.txt')
print('P@1:', result[1])
print('R@1:', result[2])

P@1: 0.8606570385653071
R@1: 0.8606570385653071


## О важности эксплоративного анализа

Но иногда пунктуация бывает и не шумом -- главное отталкиваться от задачи. Что будет если вообще не убирать пунктуацию?

In [ ]:
vec = TfidfVectorizer(ngram_range=(1, 1), tokenizer=word_tokenize)
bow = vec.fit_transform(x_train)
clf = LogisticRegression(random_state=42)
clf.fit(bow, y_train)
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


              precision    recall  f1-score   support

    negative       1.00      1.00      1.00     28048
    positive       1.00      1.00      1.00     28661

    accuracy                           1.00     56709
   macro avg       1.00      1.00      1.00     56709
weighted avg       1.00      1.00      1.00     56709



Шок! Стоило оставить пунктуацию -- и все метрики равны 1. Как это получилось? Среди неё были очень значимые токены (как вы думаете, какие?). Найдите фичи с самыми большими коэффициэнтами:

In [ ]:
sorted_weights = clf.coef_.argsort()
sorted_weights = np.flip(sorted_weights)

vec.get_feature_names_out()[sorted_weights[:, 0:20]]

array([[')', 'd', 'dd', '^_^', 'ddd', '-d', ':', '*', 'dddd', 'ddddd',
        'люблю', 'спасибо', 'ахах', 'dddddd', 'ахахах', 'х', 'okirilyuk',
        'тебе', 'мы', 'приятно']], dtype=object)

Посмотрим, как один из супер-значительных токенов справится с классификацией безо всякого машинного обучения:

In [ ]:
cool_token = '('
pred = ['positive' if not cool_token in tweet else 'negative' for tweet in x_test]
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.95      1.00      0.97     26638
    positive       1.00      0.95      0.98     30071

    accuracy                           0.97     56709
   macro avg       0.97      0.98      0.97     56709
weighted avg       0.98      0.97      0.97     56709

